# Lecture 2

### Motivation:

* What happens underneath when we call `eigh(...)`?
* How can we optimize our codes for performance by exploiting known properties of the matrices in question?
  
$\rightarrow$ explore Python documentation!
  
$\rightarrow$ Scipy modules use `lapack`, `arpack`, `blas` linear algebra libraries written in FORTRAN

## 2.1 Recap: Properties of Matrices ($\mathbb{C}^{n \times n}$)

* **Hermitian matrices:** $\quad A^\dagger := (A^*)^T = A$
  * *e.g.,* Hamiltonians, observables
  * eigenvalues are real
  * special case of $a_{ij} \in \mathbb{R} : A^T = A \rightarrow$ **symmetric matrices**

* **Unitary matrices:** $\quad A^\dagger = A^{-1} \quad (\text{or } A^\dagger A = \mathbb{1})$  
  $(\mathbb{1} = \text{identity} = \begin{pmatrix} 1 & \dots & 0 \\ \vdots & \ddots & \vdots \\ 0 & \dots & 1 \end{pmatrix})$
  * *e.g.,* time evolution operator $e^{-iHt}$, basis change
  * unit eigenvalues $|\lambda_i| = 1$
  * preserve angles and norm: $\langle \psi | \phi \rangle = \langle \psi | A^\dagger A | \phi \rangle$

* **Normal matrices:** $\quad A^\dagger A = A A^\dagger \quad$ *(commute with hermitian conjugate)*
  * *e.g.,* hermitian matrices, unitary matrices
  * all normal matrices are **diagonalizable**, i.e. there exists a unitary matrix $U$ such that $U^{-1}AU = D$ where $D$ is a diagonal matrix. $\quad (= U^\dagger A U)$

---

### Idea behind most numerical diagonalization algorithms:

Find a series of unitary transformations 

$$A \rightarrow U_1^{-1} A U_1 \rightarrow U_2^{-1} U_1^{-1} A U_1 U_2 \rightarrow \dots \rightarrow D$$

that converges to a diagonal matrix $\quad \underbrace{U_1 \dots U_n}_{:= U} = U$

Then, approximate eigenvectors will be the columns of $U$.

## 2.2 The QR algorithm

Very general, works for any normal matrix

### Basic idea: 
Iteratively calculate QR-decomposition

$$\left. \begin{aligned}
A =: A_0 &= Q_1 R_1 \\
A_1 &= R_1 Q_1
\end{aligned} \right] \text{iterate}$$

$$A_1 = Q_2 R_2$$
$$A_2 = R_2 Q_2$$
$$\vdots$$

$$\begin{aligned}
Q &\text{ unitary} \\
R &\text{ upper triangular } \begin{pmatrix} \dots & \dots \\ 0 & \dots \end{pmatrix}
\end{aligned}$$

Algorithm for calculating $Q, R$ given $A$ is known.

$$\text{(note } R_n = Q_n^{-1} A_{n-1}\text{)}$$

---

Series $A_0, A_1, A_2 \dots$ **converges to a diagonal matrix** $D$  
off-diagonal elements decay as
$$a_{ij}^{(k)} \sim \left| \frac{\lambda_i}{\lambda_j} \right|^k \qquad \begin{aligned} &\text{where wlog } i < j \text{ and} \\ &\text{sorted eigenvalues } \lambda_1 \gg \lambda_2 \gg \dots \\ &\text{are assumed.} \end{aligned}$$

Thus, one has an (approximate) **eigendecomposition**:

$$D \approx A_n = R_n Q_n = \underbrace{Q_n^{-1} A_{n-1}}_{R_n} Q_n = \dots = \underbrace{Q_n^{-1} \dots Q_1^{-1}}_{U^\dagger} A \underbrace{Q_1 \dots Q_n}_{U}$$

### Problem:
* **convergence can be slow** (if eigenvalues are spaced closely)
* **costly:** each QR-decomp. step needs $\mathcal{O}(N^3)$ operations

---

### Solution: Add a "preconditioning" step:

1) Bring $A$ to Hessenberg form $\begin{pmatrix} \dots & \dots & \dots \\ \dots & \dots & \dots \\ 0 & \dots & \dots \end{pmatrix}$ using **<u>Householder transformations</u>**.  
   *(For hermitian $A$, this is a tridiagonal form)*
2) Apply the QR iteration using **<u>Givens rotations</u>**.

---

### <u>Step 1</u>: example $4 \times 4$ matrix *(generally $N-2$ steps)*

| $A$ (Full Matrix) | $\xrightarrow{P_1^\dagger A P_1}$ | $A_1$ | $\xrightarrow{P_2^\dagger A_1 P_2}$ | Hessenberg Form |
| :---: | :---: | :---: | :---: | :---: |
| $\begin{pmatrix} \times & \times & \times & \times \\ \times & \times & \times & \times \\ \times & \times & \times & \times \\ \times & \times & \times & \times \end{pmatrix}$ | | $\begin{pmatrix} \times & \times & \times & \times \\ \mathbf{0} & \times & \times & \times \\ \mathbf{0} & \times & \times & \times \\ \mathbf{0} & \times & \times & \times \end{pmatrix}$ | | $\begin{pmatrix} \times & \times & \times & \times \\ \mathbf{0} & \times & \times & \times \\ \mathbf{0} & \mathbf{0} & \times & \times \\ \mathbf{0} & \mathbf{0} & \times & \times \end{pmatrix}$ |

<center> ⬆️ Householder trafos ("reflections") ⬆️ </center>

---

#### Construction of Householder projections:
Construct $P$ such that $\quad P^\dagger A P = \begin{pmatrix} \times & \times & \times & \dots \\ \times & \times & \times & \dots \\ 0 & \times & \times & \dots \\ 0 & 0 & \times & \dots \end{pmatrix} \quad (\vec{v} \rightarrow \vec{e}_1)$

$$P = \begin{pmatrix} 1 & 0 & \dots \\ 0 & & \\ \vdots & & \mathbb{P} \\ \vdots & & \end{pmatrix}$$

$$P\vec{v} = \alpha \vec{e}_1 = \begin{pmatrix} \alpha \\ 0 \\ \vdots \\ 0 \end{pmatrix}$$

$\mathbb{P}$ is an operation that sends $V := \begin{pmatrix} a_{21} \\ a_{31} \\ \vdots \end{pmatrix}$ to a unit vector:

$$\left( \alpha = e^{i\phi} \|V\| \text{ , where } \phi \text{ can be chosen freely} \right)$$
$$\alpha := \rho \quad \text{s.t. } \alpha \langle v | e_1 \rangle \text{ is real.}$$

choose $\quad P = \mathbb{1} - 2 |u\rangle \langle u| \qquad (= \mathbb{1} - 2 u u^\dagger)$  
where $u$ is normed $\quad \|u\| = 1 \qquad \left( \|x\| = \sqrt{\langle x | x \rangle} \right)$

$$|u\rangle = \frac{|V\rangle - \rho \|V\| |e_1\rangle}{\big\| |V\rangle - \rho \|V\| |e_1\rangle \big\|}$$

geometrically:
^ u
V  /
/_) \rho |e_1>
/\     v
/  _> e_1

---

### Check that this does the job:

* **$P$ is unitary:** $\quad P^2 = \mathbb{1} \quad \text{and} \quad P^\dagger = P \implies P^\dagger = P = P^{-1}$
  
  $$P^2 = (\mathbb{1} - 2|u\rangle\langle u|)^2 = \mathbb{1} - 4|u\rangle\langle u| + 4|u\rangle \underbrace{\langle u|u\rangle}_{1} \langle u| = \mathbb{1}$$

* **$P|V\rangle \stackrel{!}{=} \alpha |e_1\rangle$**

  $$N^2 = \big\| |V\rangle - \alpha |e_1\rangle \big\|^2 = (\langle V| - \alpha^* \langle e_1|)(|V\rangle - \alpha |e_1\rangle)$$
  
  $$\phantom{N^2} = \underbrace{\langle V|V\rangle}_{\|V\|^2 = |\alpha|^2} + |\alpha|^2 - \alpha \langle V|e_1\rangle - \underbrace{\alpha^* \langle e_1|V\rangle}_{= (\alpha \langle V|e_1\rangle)^*} \quad \begin{aligned} &\text{where } \alpha \text{ was chosen} \\ &\text{so that this is real.} \end{aligned}$$
  
  $$\phantom{N^2} = 2 \left( \langle V|V\rangle - \alpha^* \langle e_1|V\rangle \right)$$

  $$P|V\rangle = |V\rangle - 2|u\rangle\langle u|V\rangle = |V\rangle - \frac{(|V\rangle - \alpha |e_1\rangle)}{N^2} 2 \left( \langle V|V\rangle - \alpha^* \langle e_1|V\rangle \right)$$
  
  $$\phantom{P|V\rangle} = |V\rangle \left( 1 - \frac{2(\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)}{2(\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)} \right) + |e_1\rangle \left( \frac{2\alpha (\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)}{2(\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)} \right)$$
  
  $$\phantom{P|V\rangle} = \alpha |e_1\rangle \quad \checkmark$$

---

$\Rightarrow$ Applying $P$ from left sends 1st col. to $\begin{pmatrix} a_{11} \\ \alpha_{21} \\ 0 \\ \vdots \end{pmatrix}$, applying it from right leaves 1st col. unchanged.

**Next step:** Iterate procedure with $\quad \mathbb{P} = \begin{pmatrix} 1 & 0 & 0 & \dots \\ 0 & 1 & 0 & \dots \\ 0 & 0 & & \\ \vdots & \vdots & & \mathbb{P} \end{pmatrix}$

**overall scaling:** $\quad \frac{4}{3} N^3$



choose $\quad P = \mathbb{1} - 2 |u\rangle \langle u| \qquad (= \mathbb{1} - 2 u u^\dagger)$  
where $u$ is normed $\quad \|u\| = 1 \qquad \left( \|x\| = \sqrt{\langle x | x \rangle} \right)$

$$|u\rangle = \frac{|V\rangle - \rho \|V\| |e_1\rangle}{\big\| |V\rangle - \rho \|V\| |e_1\rangle \big\|}$$

| Geometric Vector Representation |
| :---: |
| $\begin{aligned} &\quad \mathbf{u} \\ &\ \nearrow \\ \mathbf{V} \rightarrow &\longrightarrow \rho |\mathbf{e}_1\rangle \\ &\ \searrow \\ &\quad \mathbf{e}_1 \end{aligned}$ |

---

### Check that this does the job:

* **$P$ is unitary:** $\quad P^2 = \mathbb{1} \quad \text{and} \quad P^\dagger = P \implies P^\dagger = P = P^{-1}$
  
  $$P^2 = (\mathbb{1} - 2|u\rangle\langle u|)^2 = \mathbb{1} - 4|u\rangle\langle u| + 4|u\rangle \underbrace{\langle u|u\rangle}_{1} \langle u| = \mathbb{1}$$

* **$P|V\rangle \stackrel{!}{=} \alpha |e_1\rangle$**

  $$N^2 = \big\| |V\rangle - \alpha |e_1\rangle \big\|^2 = (\langle V| - \alpha^* \langle e_1|)(|V\rangle - \alpha |e_1\rangle)$$
  
  $$\phantom{N^2} = \underbrace{\langle V|V\rangle}_{\|V\|^2 = |\alpha|^2} + |\alpha|^2 - \alpha \langle V|e_1\rangle - \underbrace{\alpha^* \langle e_1|V\rangle}_{= (\alpha \langle V|e_1\rangle)^*} \quad \begin{aligned} &\text{where } \alpha \text{ was chosen} \\ &\text{so that this is real.} \end{aligned}$$
  
  $$\phantom{N^2} = 2 \left( \langle V|V\rangle - \alpha^* \langle e_1|V\rangle \right)$$

  $$P|V\rangle = |V\rangle - 2|u\rangle\langle u|V\rangle = |V\rangle - \frac{(|V\rangle - \alpha |e_1\rangle)}{N^2} 2 \left( \langle V|V\rangle - \alpha^* \langle e_1|V\rangle \right)$$
  
  $$\phantom{P|V\rangle} = |V\rangle \left( 1 - \frac{2(\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)}{2(\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)} \right) + |e_1\rangle \left( \frac{2\alpha (\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)}{2(\langle V|V\rangle - \alpha^* \langle e_1|V\rangle)} \right)$$
  
  $$\phantom{P|V\rangle} = \alpha |e_1\rangle \quad \checkmark$$

---

### Matrix Transformations & Subspace Operators

| Left Multiplication Vector Mapping | Output Structural Representation | Diagonal Block Expansion ($\mathbb{P}$) |
| :---: | :---: | :---: |
| Applying operator $P$ from the left side transforms the initial tracking column to: | $\begin{pmatrix} a_{11} \\ \alpha_{21} \\ 0 \\ \vdots \end{pmatrix}$ <br><br> *(note: running the step from the right side leaves column 1 completely unaffected)* | Re-run the structural sequence down the subspace hierarchy: <br><br> $\mathbb{P} = \begin{pmatrix} 1 & 0 & 0 & \dots \\ 0 & 1 & 0 & \dots \\ 0 & 0 & & \\ \vdots & \vdots & & \mathbb{P} \end{pmatrix}$ |

<br>

**Overall Algorithmic Complexity Scaling:** $\quad \frac{4}{3} N^3$

### 2.3 Sparse matrices

The Hamiltonian matrix $H$ of physical systems is very often **sparse**, meaning that most of its elements are zero.  
Typically: **$\text{\# non-zero elements} \propto N$**

* *Example:* The finite difference Hamiltonian was tri-diagonal (cf. lecture 1), which means there are only $\sim 3N$ non-zero elements.

$$\implies \text{It would be wasteful to store all the zeros!}$$

---

### Ways to store sparse matrices:

* **Index-value pairs:** $M = \{(i_1, j_1, v_1), (i_2, j_2, v_2), \dots\}$
  * Stores: `row index`, `col index`, `value of non-zero element`.
  * **`scipy` implementation:** `coo_matrix`

* **Compressed Format Structure:** $$M = \Big\{ \big\{ (j_{11}, v_{11}), (j_{12}, v_{12}), \dots \big\} \leftarrow \text{row 1}, \ \big\{ (j_{21}, v_{21}), \dots \big\} \leftarrow \text{row 2}, \ \dots \Big\}$$
  * Stores the column index and value of each non-zero element within its respective row (e.g., the column index and value of the 1st non-zero element in row 2).
  * **`scipy` implementation:** `csr` (see also `csc`)

* **Banded matrices:** Store the $k$-th off-diagonals as vectors:
  * $k = 0$ is the main diagonal ($N$-component vector)
  * $k = \pm 1$ represents the first off-diagonals ($(N-1)$-component vector), etc.
  * **`scipy` implementation:** `dia`

---

### Important Numerical Warnings & Performance:

| Matrix Storage Rules | Computational Best Practices | Efficient Solver Algorithm |
| :---: | :---: | :---: |
| ⚠️ **Don't use sparse matrices as input to numpy functions!** | NumPy functions expect contiguous arrays and can accidentally explode the matrix back into memory-heavy dense formats. | There are very efficient methods for finding a **few eigen-pairs** of a sparse matrix $\implies$ **Lanczos algorithm**. |

### 2.4 The Lanczos algorithm

**Idea:** Find the largest / smallest eigenvalues of a very large (sparse) matrix $A = A^\dagger$ by projecting it onto a smaller subspace, the **Krylov subspace**:

$$\mathcal{K}_m = \text{span} \big\{ |v_1\rangle, A|v_1\rangle, A^2|v_1\rangle, \dots, A^{m-1}|v_1\rangle \big\} \quad \text{with } m \ll N$$

---

### Algorithmic Procedure:

Construct an orthonormal basis $\{|v_1\rangle, |v_2\rangle, \dots, |v_m\rangle\}$ of $\mathcal{K}_m$ step-by-step:

| Iterative Construction Steps | Mathematical Orthogonalization | Matrix Element Definition |
| :---: | :---: | :---: |
| **1. Choose initial state:** <br> Start with a random normed vector $|v_1\rangle$ | Set the boundary condition: <br><br> $$\beta_0 |v_0\rangle := 0$$ | Calculate diagonal elements: <br><br>$$\alpha_k = \langle v_k | A | v_k \rangle$$ |
| **2. Generate next vector:** <br> Apply $A$ and orthogonalize against previous vectors | Use the three-term recurrence relation: <br><br> $$|\tilde{v}_{k+1}\rangle = A|v_k\rangle - \alpha_k |v_k\rangle - \beta_{k-1}|v_{k-1}\rangle$$ | Calculate off-diagonal norms: <br><br>$$\beta_k = \big\| |\tilde{v}_{k+1}\rangle \big\|$$ |
| **3. Normalize:** <br> Obtain the next basis vector | Normalize the state vector: <br><br> $$|v_{k+1}\rangle = \frac{1}{\beta_k} |\tilde{v}_{k+1}\rangle$$ | *(Process repeats sequentially for $k = 1, 2, \dots, m$)* |

---

### Resulting Matrix Structure:

In this basis, the projection of $A$ onto $\mathcal{K}_m$ forms a symmetric **tridiagonal matrix** $T_m$:

$$T_m = \begin{pmatrix}
\alpha_1 & \beta_1 & 0 & \dots & 0 \\
\beta_1 & \alpha_2 & \beta_2 & \ddots & \vdots \\
0 & \beta_2 & \alpha_3 & \ddots & 0 \\
\vdots & \ddots & \ddots & \ddots & \beta_{m-1} \\
0 & \dots & 0 & \beta_{m-1} & \alpha_m
\end{pmatrix}$$

* The extreme eigenvalues of $T_m$ converge very rapidly to the extreme eigenvalues of $A$.
* **Computational Cost:** Since it only requires matrix-vector multiplications, the scaling is **$\mathcal{O}(m \cdot N)$** for sparse matrices!

## 2.4 Krylov subspace methods

### Motivation: Imaginary time evolution

$$e^{-H\tau} |\Psi_0\rangle = \sum_{k} e^{-E_k \tau} c_k |k\rangle$$

* $|\Psi_0\rangle$: some initial state (not normalized)
* $|k\rangle$: eigenstates
* $c_k = \langle k | \Psi_0 \rangle$
* $E_1 < E_2 < \dots$

$$= e^{-E_1 \tau} \Big( c_1 |1\rangle + \sum_{k=2}^{N} e^{-(E_k - E_1)\tau} c_k |k\rangle \Big)$$

* $|1\rangle$: ground state
* The summation term $\rightarrow 0$ for $\tau \rightarrow \infty$

$$\xrightarrow{\tau \rightarrow \infty} |1\rangle \quad \text{(if we keep the state normalized at any time)}$$

In principle one could use this method to find ground states by iteratively applying $e^{-H\tau} |\Psi_0\rangle$ with some fixed $\tau$ and renormalizing after each step. But how to calculate $e^{-H\tau}$, and also convergence may be slow...

Similarly, $H^k |\Psi_0\rangle$ (iteratively apply $H$) converges to the eigenvalue with largest magnitude:

$$H |\Psi_0\rangle = \sum_{k} c_k \lambda_k |k\rangle = \lambda_{\max} \sum_{k} c_k \frac{\lambda_k}{\lambda_{\max}} |k\rangle$$

* $\frac{\lambda_k}{\lambda_{\max}} < 1$ (in abs val.)
* $\rightarrow$ Portion of $|k_{\max}\rangle$ increases in each step, relative to other states.

"Matrix power method"

---

### Even more efficient: Lanczos algorithm

Build the "Krylov space" spanned by $\{|\Psi_0\rangle, H|\Psi_0\rangle, \dots, H^n|\Psi_0\rangle\} = \mathcal{K}_n(H_0)$ and represent $H$ in this subspace and diagonalize.

> **Note:** Lanczos algorithm is a special case of the Arnoldi iteration, for Hermitian matrices.

## Lanczos algorithm in detail:

Build an orthonormal basis of $\mathcal{K}_n(|\Psi_0\rangle)$ by Gram-Schmidt method and construct $h_{\mathcal{K}_n}$ on the fly:
$\{|q_1\rangle, \dots, |q_n\rangle\}$ ONB of $\mathcal{K}_n$

$$h = \begin{pmatrix} 
\langle q_1 | H | q_1 \rangle & \langle q_1 | H | q_2 \rangle & \langle q_1 | H | q_3 \rangle & \dots \\ 
\text{h.c. of} & \langle q_2 | H | q_2 \rangle & \langle q_2 | H | q_3 \rangle & \dots \\ 
\text{upper triangle} & \dots & \dots & \dots 
\end{pmatrix}$$

---

### 1st step:

* $|q_1\rangle = |\Psi_0\rangle$ (start)
* $|v_2\rangle = H |q_1\rangle$ (apply $H$)
* $|\tilde{v}_2\rangle = |v_2\rangle - |q_1\rangle \underbrace{\langle q_1 | v_2 \rangle}_{\text{1st matrix element we need: } h_{11} = \langle q_1 | H | q_1 \rangle}$ 

> **Gram-Schmidt:** subtract proj. on prev. found basis vectors

* $|q_2\rangle = \frac{|\tilde{v}_2\rangle}{\sqrt{\langle \tilde{v}_2 | \tilde{v}_2 \rangle}}$ (normalize)

---

### $n$-th step:

* $|v_n\rangle = H |q_{n-1}\rangle$
* $|\tilde{v}_n\rangle = |v_n\rangle - \sum_{k=1}^{n-1} |q_k\rangle \overbrace{\langle q_k | v_n \rangle}^{\langle q_k | H | q_{n-1} \rangle \text{ matrix element}}$

> **note:** $\langle q_k | v_n \rangle = 0$ for $k < n - 2$:
> 
> $\langle q_k | v_n \rangle = \langle q_k | H | q_{n-1} \rangle = \langle v_{k+1} | q_{n-1} \rangle$
> 
> $|q_{n-1}\rangle \perp |q_{k'}\rangle, \quad k' \le n-1$
> 
> $|v_{k+1}\rangle =$ superpos. of $|q_{k'}\rangle, \quad k' \le k+1$
> 
> $\Rightarrow \langle v_{k+1} | q_{n-1} \rangle = 0$ for $k+1 < n-1$

* $|q_n\rangle = \frac{|\tilde{v}_n\rangle}{\| |\tilde{v}_n\rangle \|}$

---

### Consequences:

1. Only need to subtract projection on **last 2 states**.
2. $h$ is **tri-diagonal**, eg. $\langle q_1 | H | q_3 \rangle = \langle q_1 | v_4 \rangle = 0$
   $\Rightarrow h$ can be diagonalized efficiently using **QR!**

## Properties of Lanczos:

* Only need to apply $H$ to states $\rightarrow$ scales well if $H$ is sparse ($\mathcal{O}(N)$).
* Not even needs $H$ explicitly but only a function that returns $H|\Psi\rangle$ upon input $|\Psi\rangle$.
* Converges fast to accurate approximation of largest magnitude eigenvalue(s).
  
  Convergence speed depends on $\frac{\lambda_2 - \lambda_1}{\lambda_N - \lambda_1}$, ratio between gap of largest to second largest eigenvalue and overall width of spectrum. Often $n \ll N$ iterations needed.

* To obtain states close to some energy $E^*$, e.g., ground states, use the **shift-invert technique**: Do Lanczos on $(H - E^*\mathbb{1})^{-1} =: G$.

  Converges to state just above $E^*$.

  * **apply $G$ to $|\Psi_k\rangle$**: solve $G^{-1} |\Psi_{k+1}\rangle = |\Psi_k\rangle$ for $|\Psi_{k+1}\rangle$ using e.g., Gauss elimination.

* Choice of initial state $|\Psi_0\rangle$ is important. Should have as large as possible ground state overlap. Typically Gaussian random vectors are used.